# AirfRANS GNN Surrogate — Kaggle GPU setup

Fallback for when Colab's free GPU quota runs out -- separate quota pool
(~30 hrs/week of P100 or T4x2). No Drive-equivalent live mount here: Kaggle
persists across sessions via **Datasets** (read-only, attached to a session)
and a notebook's own **Output** (from "Save Version"), not a synced folder.

Before running: Settings (right sidebar) > Accelerator > GPU, and
Internet > On (needed for git clone / pip install / dataset download).

## 0. This run tests the training regime, not the loss or the architecture

Two prior experiments (see `ARCHITECTURE.md` section 11 and project
history): the weighted-loss run (best checkpoint's Cd mean relative error
202.73%) and a plain-MSE run (froze partway through from a since-fixed bug,
but its one valid checkpoint, epoch 19, was *worse* than the weighted
run's equivalent). Neither got close to the AirfRANS paper's own published
baselines on this exact dataset (GraphSAGE: Cd 4.05%, Cl 0.52%).

Researched the paper's own official code
(`github.com/Extrality/AirfRANS`) instead of guessing again. Two findings:
our node features already match GraphSAGE's exactly (`[x, y, inlet_vx,
inlet_vy, sdf, normal_x, normal_y]`, just reordered) -- not the gap. But
**GraphSAGE never trains on the fixed full mesh at all**: every epoch,
every case gets a fresh random 32,000-node subsample with a spatial
radius-graph (r=0.05, max 64 neighbors) rebuilt from scratch, not the
mesh's own triangulation edges. That matters specifically because the real
mesh's near-wall edges are mostly tangential to the surface (median cosine
to the local normal ~0.0002, measured directly) -- a radius reconstruction
pulls in genuinely nearby off-wall points instead, and resampling every
epoch is free data augmentation the fixed-mesh dataset never had.

This run isolates that one variable: same architecture, same normals
feature, same distance-weighted loss (`wall_weight_peak=20`, unchanged from
the original weighted-loss run) -- only the training data regime changes,
via `src/train.py`'s new `use_radius_subsampling=True`
(`src/dataset.py`'s `CachedRadiusSubsampledDataset`,
`src/graph.py`'s `radius_graph_edges`). `subsample_n_nodes=32000`,
`subsample_r=0.05`, `subsample_max_neighbors=64` match the paper's own
published hyperparameters for GraphSAGE.

**Before running this notebook: push the local `src/graph.py`,
`src/dataset.py`, and `src/train.py` changes to GitHub**, then
`!git -C /kaggle/working/repo pull` if continuing the same session (cache
from a prior run is still valid and reusable here -- this only changes how
training *reads* the cache, not what's cached).</cell id="cell-1">

In [ ]:
# A dedicated SUBDIRECTORY per run, not just a new filename in the shared
# /kaggle/working -- os.path.dirname(checkpoint_path) is what auto-resume
# actually globs for "mgn-epoch*.ckpt", so two runs sharing a directory can
# silently cross-contaminate regardless of how differently their checkpoint
# *files* are named (see section 0 above -- this bit a real run).
#
# "_full", not "_v2": _v2 was the detect_anomaly=True diagnostic (5 epochs,
# precision="32-true", max_neighbors=8) that finally ran clean after four
# separate bugs were found and fixed (ARCHITECTURE.md section 11):
# edge_attr left in raw units, float16 target-cache overflow to inf,
# radius-graph edges crossing the solid airfoil body near the trailing
# edge, and an unbounded residual sum reaching the decoder's one
# LayerNorm-free layer. Its epoch=004 checkpoint downloaded clean (no NaN
# in any of the 804,748 parameters) -- proof the regime itself is sound,
# not proof of a trained model (only 900 real steps). This is the actual
# full-length run, starting fresh rather than resuming _v2 -- 900 steps of
# warmup isn't worth the risk of a precision-plugin state mismatch on
# resume (fp32 diagnostic -> fp16-mixed real run) when isolating from a
# known-clean state is nearly free.
CHECKPOINT_PATH = "/kaggle/working/runs/radius_subsample_full/meshgraphnet.ckpt"

import glob
import os

existing = glob.glob(os.path.join(os.path.dirname(CHECKPOINT_PATH), "mgn-epoch*.ckpt"))
print("existing checkpoints in this run's OWN directory (should be empty for a fresh start):", existing)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the repo and install dependencies

Same as Colab -- Kaggle also ships CUDA-enabled torch preinstalled, and
`torch_geometric` installs as pure Python (no `torch-scatter`/`torch-sparse`
needed, confirmed working on both machines already).

In [ ]:
REPO_URL = "https://github.com/Revanthkr1/airfrans-gnn-surrogate.git"

!git clone $REPO_URL repo
%cd repo
# Two separate fixes needed together here, not either alone:
# 1. ==2.6.1: the latest torch_geometric (2.8.0.post1) has a real circular-
#    import bug in its own __init__.py (AttributeError: partially
#    initialized module 'torch_geometric' has no attribute 'typing') that
#    reproduces reliably on Kaggle's Python 3.12 environment. 2.6.1 predates
#    the module (utils/influence.py) implicated in that import chain.
# 2. --no-deps: letting pip's resolver pull torch_geometric's own numpy has,
#    on a real run, upgraded numpy past what the environment's pre-installed
#    scipy binary was compiled against, breaking scipy.spatial.cKDTree
#    (which this project's own code needs -- src/graph.py) with an
#    ImportError unrelated to torch_geometric itself. torch_geometric's
#    actual runtime needs (torch, numpy) are already satisfied by the base
#    image, so --no-deps avoids the resolver touching them at all.
!pip install -q --no-deps torch_geometric==2.6.1
!pip install -q lightning airfrans pyvista
# `import lightning` pulls in torchmetrics, which (once transformers>=4.4 is
# detected, true in this base image) unconditionally tries to import its
# optional BERT-score text metric -- transformers -> accelerate. accelerate
# 1.13.0 has its OWN internal circular import bug (ImportError: cannot
# import name 'dispatch_model' from partially initialized module
# 'accelerate.big_modeling'), unrelated to anything in this project.
# We never use accelerate (or torchmetrics' text metrics) at all --
# transformers checks is_accelerate_available() before touching it, so
# removing accelerate entirely makes that whole broken path get skipped
# gracefully instead of erroring.
!pip uninstall -y -q accelerate

## 2. Download + preprocess, one case at a time

`af.dataset.download(unzip=True)` needs the zip (~9.34GB) *and* the fully
extracted dataset (~15GB) on disk simultaneously to extract everything at
once -- that alone exceeded this session's disk quota (`OSError: No space
left on device`, mid-extraction, before preprocessing even started).

Instead: download just the zip, then extract + cache one case at a time,
deleting each case's raw files immediately after caching. Peak disk usage
stays at roughly (zip + one case + the cache built so far) instead of
(zip + the entire raw dataset) at once. `manifest.json` doesn't need
extracting from the zip either -- it's committed to the repo at
`data/manifest.json`.

In [ ]:
from src.data import split_names
from src.preprocess import download_zip_only, stream_preprocess_from_zip

DATA_ROOT = "data"
WORK_DIR = "/kaggle/working/raw"  # transient extraction scratch, one case at a time
CACHE_DIR = "/kaggle/working/cache/full"
MANIFEST_DIR = "data"  # data/manifest.json is committed to the repo -- always present

train_names = split_names(MANIFEST_DIR, task="full", train=True)
zip_path = download_zip_only(DATA_ROOT)
stream_preprocess_from_zip(zip_path, train_names, CACHE_DIR, WORK_DIR)

import os
os.remove(zip_path)  # done with it -- frees ~9.34GB back
print(f"cached {len(os.listdir(CACHE_DIR))}/{len(train_names)} training cases")

## 3. Train, for real, on the random-subsample + radius-graph regime

`use_radius_subsampling=True` switches the training/val datasets to
`CachedRadiusSubsampledDataset` -- same on-disk cache, but every case gets
a fresh random 32,000-node subsample and a rebuilt spatial radius-graph on
every single epoch, instead of the fixed full mesh. `wall_weight_peak=20`
is unchanged from the original weighted-loss run, deliberately -- this
experiment isolates the training-data regime as the one variable.

This is the real run, not the diagnostic -- `detect_anomaly=True` is off
(it was 2-3x slower and has done its job: four bugs found, fixed, and
confirmed via a clean 5-epoch run under `precision="32-true"`), and
precision is back to `"16-mixed"` for speed. This isn't reintroducing the
earlier masking risk -- fp16 only ever hid a *real*, already-existing NaN
as a frozen-weights symptom; it never caused one. With the actual causes
fixed, mixed precision should just be faster. The frozen-weights canary
(`ARCHITECTURE.md` section 11) stays active regardless -- if weights stop
updating for 2 consecutive epochs, training halts itself automatically
with a loud printed message instead of silently wasting GPU hours. If you
see that message, don't just resume from the latest checkpoint -- resume
from the last one *before* the freeze.

`subsample_max_neighbors=8` is kept the same as the diagnostic that just
proved clean, deliberately -- one variable at a time. `max_epochs=150` is
a first real target (the paper's own GraphSAGE trains 398 epochs at this
subsampling scale, but this project's Kaggle quota is ~30 hrs/week);
`checkpoint_every_n_epochs=5` keeps resume granularity if a session runs
out of time before finishing, since `src/train.py`'s resume logic already
picks up the latest `mgn-epoch=*.ckpt` in this run's own directory.

Once this finishes (or periodically during it), evaluate with the same
paper-comparable metrics as before (`cd_mean_rel_err`, `cl_mean_rel_err`,
`cd_spearman`, `cl_spearman` in `src/evaluate.py`) against the weighted-loss
run's epoch 47 (Cd mean rel. error 202.73%) and the paper's own baselines
(Cd 4-15%). Select the checkpoint to evaluate by `val_surface_mae_mean`
(`best_surface_ckpt`), not by raw `val_loss` -- section 10.

**To persist past this session**: click "Save Version" when done (or
periodically).

In [ ]:
from src.train import main as train_main

# The real run: detect_anomaly=False (done its job -- see section 3 above)
# and precision back to "16-mixed" for speed. Everything else that the
# clean diagnostic just validated (subsample_max_neighbors=8,
# gradient_clip_val=1.0, wall_weight_peak=20) is kept unchanged --
# max_epochs and checkpoint_every_n_epochs are the only new values here.
# If this session ends before finishing, "Save Version" persists
# /kaggle/working (including this run's checkpoint directory) -- reopening
# the notebook and re-running this same cell auto-resumes from the latest
# mgn-epoch=*.ckpt (src/train.py's resume_from_checkpoint=None default).
train_main(
    dataset_root=MANIFEST_DIR,
    cache_dir=CACHE_DIR,
    stats_path="data/norm_stats.npz",
    checkpoint_path=CHECKPOINT_PATH,
    max_epochs=150,
    batch_size=1,
    accumulate_grad_batches=4,
    n_val=80,
    checkpoint_every_n_epochs=5,
    num_workers=2,
    precision="16-mixed",
    wall_weight_peak=20,
    use_radius_subsampling=True,
    subsample_n_nodes=32000,
    subsample_r=0.05,
    subsample_max_neighbors=8,
    gradient_clip_val=1.0,
    detect_anomaly=False,
)